In [1]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms


In [2]:
# debug stuff

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.13.0+cu130
True
13.0


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
from torch.utils.data import Dataset, DataLoader

# batch & normalizing

class KanjiDataset(Dataset):
    def __init__(self, image_file, label_file, indices):
        self.images = np.load(image_file, mmap_mode="r")
        self.labels = np.load(label_file, mmap_mode="r")
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]

        # load an image
        image = self.images[real_idx]
        label = self.labels[real_idx]

        # normalize
        image = image.astype(np.float32) / 255.0

        # add channel dimension: (127,128) -> (1,127,128)
        image = torch.from_numpy(image).unsqueeze(0)

        label = torch.tensor(label).long()

        return image, label

In [5]:
from sklearn.model_selection import train_test_split

# train test split
class_list = np.load("class_list.npy")
y = np.load("kanji_labels.npy")

indices = np.arange(len(y))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, stratify=y, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=y[temp_idx], random_state=42)


In [6]:
train_data = KanjiDataset("kanji_images.npy", "kanji_labels.npy", train_idx)
val_data = KanjiDataset("kanji_images.npy", "kanji_labels.npy", val_idx)
test_data = KanjiDataset( "kanji_images.npy", "kanji_labels.npy", test_idx)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

In [7]:
image, label = train_data[0]
print(image.size())
print(len(class_list))

torch.Size([1, 127, 128])
3036


In [8]:
class NeuralNet(nn.Module):

    def __init__ (self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 12, 5) # (12, 123, 124)
        self.pool = nn.MaxPool2d(2, 2) # (12, 61, 62)
        self.conv2 = nn.Conv2d(12, 24, 5) # (24, 57, 58) -> apply pool -> (24, 28, 29) -> flatten (24 * 28 * 29)

        self.fc1 = nn.Linear(24 * 28 * 29, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 3036) # 3036 kanji/character classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [9]:
net = NeuralNet().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

print(next(net.parameters()).device)  # should print cuda:0

cuda:0


In [ ]:
for epoch in range(30):
    print(f'Training epoch {epoch}...')

    running_loss = 0.0

    for i, data in enumerate(train_loader):
        inputs, labels = data

        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = net(inputs)

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 50 == 0:
            print(f'  batch {i}, loss {loss.item():.4f}')

    print(f'Loss: {running_loss / len(train_loader):.4f}')

    # validation
    net.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = net(inputs)
            loss = loss_function(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    print(f'Val Loss: {val_loss/len(val_loader):.4f}, Val Accuracy: {100*val_correct/val_total:.2f}%')
    net.train()

Training epoch 0...
  batch 0, loss 8.0193
  batch 50, loss 8.0212
  batch 100, loss 8.0180
  batch 150, loss 8.0209
  batch 200, loss 8.0154
  batch 250, loss 8.0137
  batch 300, loss 8.0122
  batch 350, loss 8.0119
  batch 400, loss 8.0176
  batch 450, loss 8.0075
  batch 500, loss 8.0176
  batch 550, loss 8.0145
  batch 600, loss 8.0161
  batch 650, loss 8.0200
  batch 700, loss 8.0138
  batch 750, loss 8.0185
  batch 800, loss 8.0226
  batch 850, loss 8.0162
  batch 900, loss 8.0279
  batch 950, loss 8.0186
  batch 1000, loss 8.0150
  batch 1050, loss 8.0266
  batch 1100, loss 8.0115
  batch 1150, loss 8.0252
  batch 1200, loss 8.0218
  batch 1250, loss 8.0171
  batch 1300, loss 8.0299
  batch 1350, loss 8.0137
  batch 1400, loss 8.0156
  batch 1450, loss 8.0165
  batch 1500, loss 8.0169
  batch 1550, loss 8.0089
  batch 1600, loss 8.0279
  batch 1650, loss 8.0202
  batch 1700, loss 8.0237
  batch 1750, loss 8.0204
  batch 1800, loss 8.0132
  batch 1850, loss 8.0179
  batch 1900, l

In [ ]:
# export model parameters
torch.save(net.state_dict(), 'kanji_model.pth')

In [ ]:
net = NeuralNet().to(device)
net.load_state_dict(torch.load('kanji_model.pth', map_location=device))

In [ ]:
# eval on test data

correct = 0
total = 0

net.eval()

# feed data in to get predictions and compare to labels
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f'Accuracy: {accuracy}%')